In [1]:
import pandas as pd
import numpy as np

# para viz
import altair as alt

# para el calculo del IVF
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# para el calculo del PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [3]:
df = pd.read_csv(
    'ds_source/indicadores_parroquiales_seleccion.csv',
    encoding='latin1',
    sep=';',
    decimal=','
)

df = df.rename(columns={'p_año': 'p_anio'})

df.columns = df.columns.str.lower()

df['p_ciudad'] = df['p_ciudad'].astype('Int64').astype(str).str.zfill(6)

In [4]:
df.head()

,p_anio,p_ciudad,provincia,canton,parroquia,tasa_participacion_global,tasa_participacion_bruta,tasa_desempleo,empleo_total,empleo_formal,...,nini,desempleo_juvenil,trabajo_infantil,empleo_manufactura,tasa_asistencia_clases,pobreza_ingresos,pobreza_extrema_ingresos,pobreza_multidimensional,pobreza_extrema_multidimensional,necesidades_basicas_insatisfechas
0,2007,010150,AZUAY,CUENCA,CUENCA,67.863567,49.393567,5.558733,94.441267,69.034833,...,9.627167,9.572733,4.534000,18.929133,81.170667,12.027167,3.225633,11.9836,0.6598,8.4357
1,2007,070150,EL ORO,MACHALA,MACHALA,69.333167,47.943867,5.358800,94.641200,47.899267,...,18.096067,8.495700,4.630767,8.986300,73.298600,24.675667,7.085300,34.2322,10.0497,34.9903
2,2007,090150,GUAYAS,GUAYAQUIL,GUAYAQUIL,69.575300,49.169967,7.754667,92.245333,52.301900,...,17.493100,13.822567,4.912333,13.204367,75.789400,20.251200,5.591267,27.2071,7.8995,33.7344
3,2007,170150,PICHINCHA,DISTRITO METROPOLITANO DE QUITO,QUITO,68.276200,50.522900,6.525067,93.474933,67.801000,...,10.992300,11.136133,3.407133,15.148067,79.390200,10.737867,3.262500,9.6647,1.4425,0.1811
4,2007,180150,TUNGURAHUA,AMBATO,AMBATO,68.935600,50.985967,4.747567,95.252433,66.567567,...,9.522433,9.044567,3.967133,20.620200,83.773800,15.988000,4.482433,13.3385,2.4116,14.4037


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 27 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   p_anio                             95 non-null     int64  
 1   p_ciudad                           95 non-null     str    
 2   provincia                          95 non-null     str    
 3   canton                             95 non-null     str    
 4   parroquia                          95 non-null     str    
 5   tasa_participacion_global          95 non-null     float64
 6   tasa_participacion_bruta           95 non-null     float64
 7   tasa_desempleo                     95 non-null     float64
 8   empleo_total                       95 non-null     float64
 9   empleo_formal                      95 non-null     float64
 10  empleo_informal                    95 non-null     float64
 11  empleo_adecuado                    95 non-null     float64
 12  subempl

In [6]:
df.describe()

,p_anio,tasa_participacion_global,tasa_participacion_bruta,tasa_desempleo,empleo_total,empleo_formal,empleo_informal,empleo_adecuado,subempleo,no_remunerado,...,nini,desempleo_juvenil,trabajo_infantil,empleo_manufactura,tasa_asistencia_clases,pobreza_ingresos,pobreza_extrema_ingresos,pobreza_multidimensional,pobreza_extrema_multidimensional,necesidades_basicas_insatisfechas
count,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,...,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000
mean,2016.000000,65.505139,48.779789,5.678801,94.321199,65.257531,28.706685,53.307862,14.177952,5.387865,...,15.387821,11.142814,1.762461,14.903995,79.365237,11.935566,3.094450,14.888899,2.700926,14.581446
std,5.506283,3.180622,2.610130,2.267723,2.267723,8.987805,8.440437,7.719434,5.930425,1.970724,...,4.479938,3.889055,1.652124,4.173378,3.459475,4.821898,1.705958,8.231461,2.990140,9.642452
min,2007.000000,57.972222,41.599200,3.027550,84.467725,46.570125,16.678875,32.631825,3.959750,2.193511,...,7.429200,4.943000,0.000000,7.216525,73.298600,3.974925,0.794025,3.942425,0.018544,0.181100
25%,2011.000000,63.336689,47.495613,4.100521,93.468992,57.476258,21.884538,47.956078,9.940213,3.891867,...,11.450071,8.519881,0.525936,12.479425,75.699763,8.914613,2.108213,8.056447,0.756500,6.548179
50%,2016.000000,64.989700,48.568225,4.885050,95.114950,68.893200,25.516511,52.815500,12.719150,4.996350,...,15.117975,9.937150,1.134125,14.677917,80.351900,10.909300,2.693450,12.886700,1.748558,10.159175
75%,2021.000000,67.740779,50.377979,6.531008,95.899479,72.601062,35.967175,59.115475,18.571150,6.355738,...,19.745213,13.304525,2.434942,18.753979,82.180306,14.118350,3.817450,21.146058,3.392150,23.275312
max,2025.000000,72.671075,55.750500,15.532275,96.972450,79.398100,46.857217,70.548075,30.546450,10.305525,...,23.070275,29.503900,6.855600,21.609775,86.736489,26.839550,10.156625,39.656400,17.684500,41.361400


In [7]:
id_cols = ['p_anio', 'p_ciudad', 'provincia', 'canton', 'parroquia']
num_cols = [c for c in df.columns if c not in id_cols]

In [8]:
df[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
tasa_participacion_global,95.0,65.505139,3.180622,57.972222,63.336689,64.989700,67.740779,72.671075
tasa_participacion_bruta,95.0,48.779789,2.610130,41.599200,47.495613,48.568225,50.377979,55.750500
tasa_desempleo,95.0,5.678801,2.267723,3.027550,4.100521,4.885050,6.531008,15.532275
empleo_total,95.0,94.321199,2.267723,84.467725,93.468992,95.114950,95.899479,96.972450
empleo_formal,95.0,65.257531,8.987805,46.570125,57.476258,68.893200,72.601062,79.398100
empleo_informal,95.0,28.706685,8.440437,16.678875,21.884538,25.516511,35.967175,46.857217
empleo_adecuado,95.0,53.307862,7.719434,32.631825,47.956078,52.815500,59.115475,70.548075
subempleo,95.0,14.177952,5.930425,3.959750,9.940213,12.719150,18.571150,30.546450
no_remunerado,95.0,5.387865,1.970724,2.193511,3.891867,4.996350,6.355738,10.305525
otro_no_pleno,95.0,20.527407,3.436197,13.258233,17.932075,20.204200,22.678112,29.938558


# EDA

In [9]:
df_long = df[num_cols].melt(var_name='variable', value_name='valor')

In [10]:
chart_hist = alt.Chart(df_long).mark_bar().encode(
    x=alt.X(
        'valor:Q',
        bin=alt.Bin(maxbins=30),
        scale=alt.Scale(domain=[0, 100]),
        title='Porcentaje (%)',
        axis=alt.Axis(format='.0f', labelExpr="datum.label + '%'")
    ),
    y=alt.Y('count():Q', title='Frecuencia')
).properties(
    width=180,
    height=120
).facet(
    facet=alt.Facet('variable:N', title='Distribuciones por variable'),
    columns=5
)

chart_hist

alt.FacetChart(...)

In [11]:
orden_mediana = df_long.groupby('variable')['valor'].median().sort_values().index.tolist()

chart_box = alt.Chart(df_long).mark_boxplot().encode(
    x=alt.X(
        'valor:Q',
        scale=alt.Scale(domain=[0, 100]),
        title='Porcentaje (%)',
        axis=alt.Axis(format='.0f', labelExpr="datum.label + '%'")
    ),
    y=alt.Y(
        'variable:N',
        sort=orden_mediana,
        title=None
    )
).properties(
    width=700,
    height=500,
    title='Boxplots por variable'
)

chart_box

alt.Chart(...)

# Correlación de Pearson

In [12]:
corr = df[num_cols].corr(method='pearson')

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

corr_pairs = (
    corr.where(mask)
        .stack()
        .reset_index()
)

corr_pairs.columns = ['variable_1', 'variable_2', 'pearson_r']
corr_pairs['abs_r'] = corr_pairs['pearson_r'].abs()

corr_pairs = corr_pairs.sort_values('abs_r', ascending=False).reset_index(drop=True)

corr_pairs.head(30)

,variable_1,variable_2,pearson_r,abs_r
0,tasa_desempleo,empleo_total,-1.000000,1.000000
1,empleo_formal,empleo_informal,-0.979382,0.979382
2,empleo_total,desempleo_juvenil,-0.951044,0.951044
3,tasa_desempleo,desempleo_juvenil,0.951044,0.951044
4,pobreza_multidimensional,necesidades_basicas_insatisfechas,0.948968,0.948968
5,empleo_adecuado,subempleo,-0.906090,0.906090
6,pobreza_multidimensional,pobreza_extrema_multidimensional,0.898964,0.898964
7,pobreza_ingresos,pobreza_extrema_ingresos,0.881455,0.881455
8,empleo_formal,tasa_asistencia_clases,0.859146,0.859146
9,pobreza_extrema_multidimensional,necesidades_basicas_insatisfechas,0.845212,0.845212


In [13]:
corr = df[num_cols].corr(method='pearson')

corr_long = (
    corr.reset_index()
    .melt(id_vars='index', var_name='variable_2', value_name='correlacion')
    .rename(columns={'index': 'variable_1'})
)

orden_vars = corr.columns.tolist()
idx_map = {var: i for i, var in enumerate(orden_vars)}

corr_long['i'] = corr_long['variable_1'].map(idx_map)
corr_long['j'] = corr_long['variable_2'].map(idx_map)

# solo triángulo inferior
corr_long_inf = corr_long[corr_long['i'] < corr_long['j']]

base = alt.Chart(corr_long_inf).encode(
    x=alt.X('variable_1:N', sort=orden_vars, title=None),
    y=alt.Y('variable_2:N', sort=orden_vars, title=None),
    tooltip=[
        alt.Tooltip('variable_1:N', title='Variable 1'),
        alt.Tooltip('variable_2:N', title='Variable 2'),
        alt.Tooltip('correlacion:Q', format='.3f', title='Correlación')
    ]
)

heatmap = base.mark_rect().encode(
    color=alt.Color(
        'correlacion:Q',
        scale=alt.Scale(
            domain=[-1, 0, 1],
            range=['#b2182b', '#f7f7f7', '#2166ac']
        ),
        title='r de Pearson'
    )
)

text = base.mark_text(fontSize=10).encode(
    text=alt.Text('correlacion:Q', format='.2f'),
    color=alt.condition(
        'abs(datum.correlacion) >= 0.5',
        alt.value('white'),
        alt.value('black')
    )
)

chart = (heatmap + text).properties(
    width=600,
    height=600,
    title='Matriz de correlación de Pearson'
).configure_view(
    strokeWidth=0
)

chart

alt.LayerChart(...)

# Variance Inflation Factor (VIF)

In [14]:
exclude_cols = ['p_anio', 'p_ciudad', 'provincia', 'canton', 'parroquia']
X = df.drop(columns=exclude_cols).copy()
X = X.apply(pd.to_numeric, errors='coerce').dropna()

X_const = sm.add_constant(X)

vif_df = pd.DataFrame({
    'variable': X_const.columns,
    'VIF': [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
})

vif_df['tolerancia'] = 1 / vif_df['VIF']
vif_df

/Users/erickedu85/Dropbox/python_projects/enemdu_icm/.venv/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,variable,VIF,tolerancia
0,const,inf,0.000000
1,tasa_participacion_global,16.221656,0.061646
2,tasa_participacion_bruta,25.074239,0.039882
3,tasa_desempleo,inf,0.000000
4,empleo_total,inf,0.000000
5,empleo_formal,79.765386,0.012537
6,empleo_informal,72.236874,0.013843
7,empleo_adecuado,106.539948,0.009386
8,subempleo,65.496673,0.015268
9,no_remunerado,18.769322,0.053278


# PCA

In [15]:
# se elimina empleo_total por alta colinealidad con tasa_desempleo
cols_excluir_pca = [
    'p_anio', 'p_ciudad', 'provincia', 'canton', 'parroquia', 'empleo_total'
]

X = df.drop(columns=cols_excluir_pca).copy()
X = X.apply(pd.to_numeric, errors='coerce').dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

## Varianza

In [16]:
explained_variance = pd.DataFrame({
    'componente': [f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],
    'varianza_explicada': pca.explained_variance_ratio_,
    'varianza_acumulada': pca.explained_variance_ratio_.cumsum()
})

explained_variance

,componente,varianza_explicada,varianza_acumulada
0,PC1,0.385731,0.385731
1,PC2,0.207440,0.593171
2,PC3,0.150986,0.744157
3,PC4,0.098188,0.842345
4,PC5,0.043597,0.885942
5,PC6,0.028923,0.914865
6,PC7,0.023633,0.938499
7,PC8,0.016172,0.954670
8,PC9,0.010794,0.965464
9,PC10,0.008235,0.973699


## Cargas para PC1

In [17]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(X.shape[1])],
    index=X.columns
)

loadings[['PC1']].sort_values('PC1')

,PC1
empleo_informal,-0.316933
pobreza_multidimensional,-0.301161
necesidades_basicas_insatisfechas,-0.297699
nini,-0.288538
pobreza_extrema_multidimensional,-0.257277
pobreza_ingresos,-0.251358
subempleo,-0.217174
brecha_adecuado_hm,-0.175644
pobreza_extrema_ingresos,-0.145579
desempleo_juvenil,-0.138268


## Construir el Indice Compuesto Multivariado (ICM) solo con el PC1

 - 0 = peor ICM relativo observado
 - 100 = mejor ICM relativo observado

In [18]:
indice_compuesto_multivariado = X_pca[:, 0]

indice_df = df.loc[X.index, ['p_anio', 'p_ciudad', 'provincia', 'canton', 'parroquia']].copy()
indice_df['indice_compuesto_multivariado'] = indice_compuesto_multivariado

indice_min = indice_df['indice_compuesto_multivariado'].min()
indice_max = indice_df['indice_compuesto_multivariado'].max()

# Transformación min-max del indice
indice_df['indice_compuesto_multivariado_0_100'] = (
    (indice_df['indice_compuesto_multivariado'] - indice_min) / (indice_max - indice_min)
) * 100

indice_df

,p_anio,p_ciudad,provincia,canton,parroquia,indice_compuesto_multivariado,indice_compuesto_multivariado_0_100
0,2007,010150,AZUAY,CUENCA,CUENCA,2.221896,81.303897
1,2007,070150,EL ORO,MACHALA,MACHALA,-5.623312,3.960425
2,2007,090150,GUAYAS,GUAYAQUIL,GUAYAQUIL,-3.782332,22.110071
3,2007,170150,PICHINCHA,DISTRITO METROPOLITANO DE QUITO,QUITO,2.116305,80.262905
4,2007,180150,TUNGURAHUA,AMBATO,AMBATO,1.519539,74.379575
...,...,...,...,...,...,...,...
90,2025,010150,AZUAY,CUENCA,CUENCA,2.550913,84.547566
91,2025,070150,EL ORO,MACHALA,MACHALA,-2.504504,34.707786
92,2025,090150,GUAYAS,GUAYAQUIL,GUAYAQUIL,-3.029002,29.536922
93,2025,170150,PICHINCHA,DISTRITO METROPOLITANO DE QUITO,QUITO,0.962578,68.888669


## Series temporales del Indice Compuesto Multivariado

In [19]:
indice_df['anio_fecha'] = pd.to_datetime(indice_df['p_anio'].astype(str), format='%Y')

chart = alt.Chart(indice_df).mark_line(point=True).encode(
    x=alt.X(
        'anio_fecha:T',
        title='Año',
        scale=alt.Scale(nice=False),
        axis=alt.Axis(format='%Y')
    ),
    y=alt.Y(
        'indice_compuesto_multivariado_0_100:Q', 
        title='Índice Compuesto Multivariado (0-100)',
        axis=alt.Axis(format='.0f', labelExpr="datum.label + '%'")
    ),
    color=alt.Color(
        'parroquia:N', 
        title='Parroquia'
    ),
    tooltip=[
        alt.Tooltip('p_anio:O', title='Año'),
        alt.Tooltip('provincia:N'),
        alt.Tooltip('canton:N'),
        alt.Tooltip('parroquia:N'),
        alt.Tooltip('indice_compuesto_multivariado_0_100:Q', format='.2f', title='Índice')
    ]
).properties(
    width=700,
    height=400,
    title='Evolución del Índice Compuesto Multivariado (ICM) a lo largo del tiempo'
)

chart

alt.Chart(...)